# LD-2 — FUMÉE Kaggle CPU : tests, T/E/EW/FREEZE/CAL sur mf_0000, EVAL sur pages de remplacement

In [ ]:

import time
T_START = time.monotonic()          # pris au tout début de la cellule, avant la copie du kit
import datetime, faulthandler, hashlib, json, os, shlex, shutil, subprocess, sys
from pathlib import Path
faulthandler.enable()

def _utc():
    return datetime.datetime.now(datetime.timezone.utc).strftime('%H:%M:%S')

print('ld2 start', _utc(), flush=True)
OUT = Path('/kaggle/working')
KIT_CANDIDATES = ['/kaggle/input/babaorum-ld2-kit', '/kaggle/input/datasets/mabaku/babaorum-ld2-kit']
PAGES_CANDIDATES = ['/kaggle/input/babaorum-e2e3-run3-fp-classes/e2e3/mf', '/kaggle/input/notebooks/mabaku/babaorum-e2e3-run3-fp-classes/e2e3/mf']
CKPT_CANDIDATES = ['/kaggle/input/babaorum-e2e5-lambda/logs/checkpoint_last.pt', '/kaggle/input/notebooks/mabaku/babaorum-e2e5-lambda/logs/checkpoint_last.pt']
LD1_CANDIDATES = ['/kaggle/input/babaorum-ld1-run/ld1/ld1_step001871.pt', '/kaggle/input/notebooks/mabaku/babaorum-ld1-run/ld1/ld1_step001871.pt']
REF_CANDIDATES = ['/kaggle/input/babaorum-e2e5-lambda/e2e5_lambda_run_report.json', '/kaggle/input/notebooks/mabaku/babaorum-e2e5-lambda/e2e5_lambda_run_report.json']
REF_PREFIX = '3abfe0d7'
EST_FACTOR = 1.5
INIT_SHA = 'ef5e826803c347f73e49c83c0f21f61399f9ea06ee985905bd17644dff3844fa'
INIT_STEP = 2000
DEC_SHA = 'a1c5fbab288c99e4911789f926d2d568815fd2a165310da5020ac7c140f8197b'
DEC_STEP = 1871
TIMEOUTS = {'D': 900, 'T': 2700, 'E': 9000, 'EW': 4800, 'FREEZE': 300, 'CAL': 900, 'pytest_heavy': 5400, 'TW': 1500}
LIMITS = {'deadline_s': 19800, 'tail_reserve_s': 2400, 'stop_margin_s': 120, 'eval_margin_s': 300, 'ew_grace_s': 180}
def resolve_one(candidates, marker):
    """Exactement un candidat dont <candidat>/<marqueur> existe. Aucun listage."""
    hits = [Path(c) for c in candidates if (Path(c) / marker).is_file()]
    if len(hits) != 1:
        raise RuntimeError(f"montage : {len(hits)} candidat(s) pour {marker}")
    return hits[0]

def resolve_file(candidates):
    """Exactement un candidat qui est un fichier (§12 (15))."""
    hits = [Path(c) for c in candidates if Path(c).is_file()]
    if len(hits) != 1:
        raise RuntimeError(f"fichier : {len(hits)} candidat(s)")
    return hits[0]


def sha256(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()

KIT = resolve_one(KIT_CANDIDATES, "config/ld2.json")
PAGES = resolve_one(PAGES_CANDIDATES, "pages/mf_0000.json")
CKPT = resolve_file(CKPT_CANDIDATES)
LD1CK = resolve_file(LD1_CANDIDATES)
assert sha256(CKPT) == INIT_SHA, "checkpoint de l'encodeur (E2E-5) : sha différent"
assert sha256(LD1CK) == DEC_SHA, "checkpoint du décodeur (LD-1) : sha différent"
sys.path.insert(0, str(KIT / "src"))
sys.path.insert(0, str(KIT / "scripts"))
import importlib.util as _u
_spec = _u.spec_from_file_location("data2_common", KIT / "scripts" / "data2_common.py")
C = _u.module_from_spec(_spec); _spec.loader.exec_module(C)
import ld2_core as C2

def _argv(phase, extra=()):
    return [sys.executable, str(KIT / "scripts" / "train_ld2.py"), "--phase", phase,
            "--root", str(PAGES), "--init-checkpoint", str(CKPT), "--decoder-checkpoint", str(LD1CK),
            "--device", "cpu", "--cache-dir", "/tmp/ld2_hf", "--out", str(RUN), *extra]

def wrapped(ph, argv, inner_s):
    """Étapes du témoin (TW, EW) : le code de sortie va dans <RUN>/../exits/ld2_<ph>_rc.txt et l'étape rend TOUJOURS 0 au carnet,
    même après un arrêt dur du processus (OOM de l'hôte, segfault). `timeout -s KILL inner_s` tue la commande AVANT le délai de
    l'étape du carnet (inner_s entre le chien de garde et ce délai), pour que sh survive, écrive 137 et rende 0 ; FREEZE, CAL
    et EVAL relisent ce fichier."""
    exits = RUN.parent / "exits"
    exits.mkdir(parents=True, exist_ok=True)
    rc = shlex.quote(str(exits / f"ld2_{ph}_rc.txt"))
    cmd = shlex.join(["timeout", "-s", "KILL", f"{inner_s:.1f}", *[str(a) for a in argv]])
    return ["sh", "-c", cmd + f"; echo $? > {rc}; exit 0"]

SMOKE = OUT / "ld2_smoke"
RUN = SMOKE / "ld2"
CAL_OUT = SMOKE / "ld2_cal"
SMOKE.mkdir(parents=True, exist_ok=True)
DECOY_PAGES = ['mf_0000', 'mf_0001']
HEAVY_ENV = dict(os.environ, LD2_HEAVY="1", PYTHONPATH=str(KIT / "src") + os.pathsep + str(KIT / "scripts"))
HEAVY_ENV["HF_HUB_OFFLINE"] = "1"
OFFLINE = lambda: dict(os.environ, HF_HUB_OFFLINE="1")
DECOY_REF = SMOKE / "decoy_reference.json"

def s_decoy_reference():
    """Référence FACTICE pour les pages de remplacement : le bootstrap et le critère s'exécutent, sans aucune valeur réelle."""
    DECOY_REF.write_text(json.dumps({"eval": [{"page_id": p, "metrics": {"after": {"recall": 0.9, "precision": 0.7, "n_pred": 10}}}
                                              for p in DECOY_PAGES]}))

def _smoke_argv(phase, extra=()):
    return _argv(phase, ["--pages", "mf_0000", *extra]) if phase in ("T", "TW", "E", "EW") else _argv(phase, extra)

STEPS = [
    {"name": "D", "argv": [sys.executable, str(KIT / "scripts" / "train_ld2.py"), "--phase", "D", "--out", str(RUN),
                            "--cache-dir", "/tmp/ld2_hf"], "timeout_s": TIMEOUTS["D"]},
    # TOUS les tests des deux fichiers, sans -k ; rc écrit dans un fichier, sans bloquer la suite
    {"name": "pytest_heavy", "argv": ["sh", "-c", "cd \"" + str(KIT) + "\" && " + sys.executable + " -m pytest -q -rs "
                                      "tests/test_ld2.py tests/test_ld2_heavy.py > \"" + str(SMOKE) + "/ld2_pytest.txt\" 2>&1; "
                                      "echo $? > \"" + str(SMOKE) + "/ld2_pytest_rc.txt\"; exit 0"],
     "env": HEAVY_ENV, "timeout_s": TIMEOUTS["pytest_heavy"]},
    {"name": "decoy", "fn": s_decoy_reference},
    {"name": "T", "argv": _smoke_argv("T", ["--timing-steps", "3", "--max-steps", "20"]), "timeout_s": TIMEOUTS["T"], "env": OFFLINE},
    {"name": "TW", "argv": lambda: wrapped("TW", _smoke_argv("TW", ["--timing-steps", "3", "--deadline-s", "1200"]), 1200 + 0.75 * LIMITS["ew_grace_s"]), "timeout_s": TIMEOUTS["TW"],
     "env": OFFLINE},
    {"name": "E", "argv": _smoke_argv("E"), "timeout_s": TIMEOUTS["E"], "env": OFFLINE},
    {"name": "EW", "argv": lambda: wrapped("EW", _smoke_argv("EW", ["--deadline-s", "1200"]), 1200 + 0.75 * LIMITS["ew_grace_s"]), "timeout_s": TIMEOUTS["EW"],
     "env": OFFLINE},
    {"name": "FREEZE", "argv": _smoke_argv("FREEZE"), "timeout_s": TIMEOUTS["FREEZE"], "env": OFFLINE},
    {"name": "CAL", "argv": _smoke_argv("CAL", ["--pages", "mf_0000", "--smoke", "--eval-pages", *DECOY_PAGES,
                                                "--cal-out", str(CAL_OUT)]), "timeout_s": 5400, "env": OFFLINE},
    {"name": "EVAL", "argv": _smoke_argv("EVAL", ["--smoke", "--eval-pages", *DECOY_PAGES, "--reference-report", str(DECOY_REF)]),
     "timeout_s": 10800, "env": OFFLINE},
]
SUMMARY = {"kind": "SMOKE_KAGGLE_CPU_VRAIS_CHECKPOINTS", "init_sha256": INIT_SHA, "decoder_sha256": DEC_SHA}
EXIT = None
try:
    EXIT = C.execute_pipeline(SMOKE, STEPS, heartbeat_interval_s=60.0)
finally:
    for f in RUN.glob("ld2_*_step*.pt"):                         # la fumée ne rapporte que du JSON
        f.unlink()
    SUMMARY["pipeline"] = {k: EXIT[k] for k in ("status", "step", "exit_code", "durations_s")} if EXIT else None
    for name in ("ld2_pytest.txt", "ld2_pytest_rc.txt"):
        f = SMOKE / name
        if f.is_file():
            SUMMARY[name] = f.read_text()[-6000:]
    for name in ("ld2_manifest.json", "ld2_witness_status.json", "ld2_report.json", "ld2_main_train_log.jsonl",
                 "ld2_witness_train_log.jsonl"):
        f = RUN / name
        if f.is_file():
            SUMMARY[name] = f.read_text()[-20000:] if name.endswith(".jsonl") else json.loads(f.read_text())
    cal = CAL_OUT / "ld2_cal.json"
    if cal.is_file():
        SUMMARY["ld2_cal.json"] = json.loads(cal.read_text())
    (OUT / "ld2_smoke_summary.json").write_text(json.dumps(SUMMARY, indent=1, ensure_ascii=False, default=str))
    shutil.rmtree("/tmp/ld2_hf", ignore_errors=True)
print("SMOKE", SUMMARY.get("pipeline"), flush=True)
